# سبق ۰۵ - ایجنٹک RAG


## ترتیب

یہ نوٹ بک Microsoft Agent Framework کا استعمال کرتے ہوئے Agentic RAG (Retrieval-Augmented Generation) پیٹرن کو ظاہر کرتی ہے۔

**ضروریات:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — آپ کا Azure AI سرچ سروس اینڈ پوائنٹ
- `AZURE_SEARCH_API_KEY` — آپ کی Azure AI سرچ API کلید
- ماحول کے متغیرات کے ذریعے ترتیب دیا گیا Azure OpenAI تعیناتی
- Azure CLI کی توثیق شدہ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ایجنٹک RAG کیا ہے؟

روایتی RAG ایک مقررہ طریقہ کار کی پیروی کرتا ہے: دستاویزات حاصل کریں، پھر جواب تیار کریں۔ **ایجنٹک RAG** اس سے آگے جاتا ہے اور ایجنٹ کو خودمختاری دیتا ہے کہ وہ فیصلہ کرے کہ معلومات **کب** اور **کیسے** حاصل کی جائیں۔

ایجنٹک RAG کے ساتھ، ایجنٹ کر سکتا ہے:
- **فیصلہ** کرے کہ سوال کا جواب دینے سے پہلے معلومات حاصل کرنا ضروری ہے یا نہیں
- **چُن** سکے کہ کون سا ڈیٹا سورس یا آلہ استفسار کے لیے استعمال کرنا ہے
- **جانچے** حاصل کردہ نتائج کو اور اگر پہلا کوشش ناکافی ہو تو مزید معلومات حاصل کرے
- **معلومات کو مربوط** جواب میں چند حاصل کرنے کے مراحل سے جمع کرے

یہ ایجنٹ کو ایک جامد حاصل کریں پھر جواب تیار کریں طریقہ کار کے مقابلے میں زیادہ لچکدار اور درست بناتا ہے۔


## سرچ ٹول بنانا

ایجنٹک RAG میں، بیرونی ڈیٹا ذرائع کو **ٹولز** کے طور پر لپیٹا جاتا ہے جنہیں ایجنٹ ضرورت کے مطابق استعمال کر سکتا ہے۔ اس سے ایجنٹ کو بازیافت کو محض ایک اور عمل کے طور پر لینے کی اجازت ملتی ہے جسے وہ انجام دے سکتا ہے، نہ کہ ایک لازمی قدم کے طور پر۔

نیچے ہم ایک سفر کا نالج بیس تیار کرتے ہیں اور اسے ایک ٹول کے طور پر پیش کرتے ہیں جسے ایجنٹ منزل کی معلومات دیکھنے کے لیے بلا سکتا ہے۔


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ایجنٹ کی تعمیر

اب ہم ایک ایسا ایجنٹ بناتے ہیں جسے ہدایت دی گئی ہے کہ **ہمیشہ جواب دینے سے پہلے معلومات حاصل کرے**۔ ایجنٹ اپنے جوابات کو اپنی تربیتی ڈیٹا پر انحصار کرنے کے بجائے علمی بنیاد میں مضبوط کرنے کے لیے `search_travel_knowledge` ٹول استعمال کرتا ہے۔


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## تکراری بازیافت — میکر-چیکر پیٹرن

Agentic RAG کا ایک اہم فائدہ **تکراری بازیافت** ہے۔ ایجنٹ متعدد بار تلاش کر سکتا ہے تاکہ اپنی ابتدائی دریافتوں کی تصدیق، تصحیح یا توسیع کر سکے — بالکل "میکر-چیکر" ورک فلو کی طرح:

1. **میکر قدم**: ایجنٹ ابتدائی معلومات حاصل کرتا ہے اور جواب کا مسودہ تیار کرتا ہے۔
2. **چیکر قدم**: ایجنٹ تفصیلات کی تصدیق یا خلا کو پُر کرنے کے لیے اضافی تلاش کرتا ہے۔

نیچے، ایجنٹ سے ایک ایسا سوال پوچھا گیا ہے جس میں متعدد مقامات کا موازنہ کرنا ہوتا ہے، جس کی بنا پر اسے کئی بار تلاش کرنا پڑتا ہے۔


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ مائیکروسافٹ ایجنٹ فریم ورک کا استعمال کرتے ہوئے **ایجنٹک RAG** سسٹم کیسے بنایا جائے:

- **ایجنٹک RAG** ایجنٹس کو خود مختارانہ طور پر فیصلہ کرنے دیتا ہے کہ کب معلومات حاصل کی جائیں، جس سے بازیابی متحرک ہوتی ہے نہ کہ مقررہ۔
- **ٹولز بطور ڈیٹا ذرائع**: خارجی علم کے ذخیرے (جیسے Azure AI Search) کو ایسے ٹولز کے طور پر لپیٹا جاتا ہے جنہیں ایجنٹ بلا سکتا ہے۔
- **تکراری بازیابی**: میکر-چیکر پیٹرن ایجنٹ کو کئی بازیابی کے دور انجام دینے کے قابل بناتا ہے — تلاش کرنا، تصدیق کرنا، اور بہتر بنانا — اس سے پہلے کہ حتمی جواب تیار کیا جائے۔

پیداوار میں، آپ `TRAVEL_KNOWLEDGE_BASE` کو ایک حقیقی Azure AI Search انڈیکس سے تبدیل کریں گے تاکہ بڑے پیمانے پر سفر کے دستاویزات کی بازیابی کو سنبھالا جا سکے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دستخطی اعلامیہ**:
اس دستاویز کا ترجمہ AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے کیا گیا ہے۔ ہم درستگی کے لیے کوشاں ہیں، لیکن براہ کرم آگاہ رہیں کہ خودکار تراجم میں غلطیاں یا کمی بیشی ہو سکتی ہے۔ اصل دستاویز اپنی مادری زبان میں ہی معتبر ماخذ سمجھی جائے گی۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کا مشورہ دیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والے کسی بھی غلط فہمی یا غلط تشریحات کے لیے ہم ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
